# **Section 3: Tuning**

## **Introduction**
In this section, we will explore how to fine-tune federated learning models using configurable parameters. We will use the Flower framework to implement client-server communication and tune hyperparameters dynamically.

---

## **Step 1: Import Required Libraries**


In [ ]:
pip install -r requirements.txt

In [ ]:
from flwr.client import Client, ClientApp, NumPyClient
from flwr.server import ServerApp, ServerConfig
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset

from utils3 import *

---

## **Step 2: Prepare the Datasets**
We will use Flower Datasets to load and preprocess the MNIST dataset in a federated learning setting.

### **Load Federated Dataset**


In [ ]:
def load_data(partition_id):
    fds = FederatedDataset(dataset="mnist", partitioners={"train": 5})
    partition = fds.load_partition(partition_id)

    traintest = partition.train_test_split(test_size=0.2, seed=42)
    traintest = traintest.with_transform(normalize)
    trainset, testset = traintest["train"], traintest["test"]

    trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
    testloader = DataLoader(testset, batch_size=64)
    return trainloader, testloader

---

## **Step 3: Configure Clients**
We will define how clients receive and process configuration parameters.

### **Define Fit Configuration**


In [ ]:
def fit_config(server_round: int):
    config_dict = {
        "local_epochs": 2 if server_round < 3 else 5,
    }
    return config_dict

### **Define Federated Averaging Strategy in Server Function**


In [ ]:
net = SimpleModel()
params = ndarrays_to_parameters(get_weights(net))

def server_fn(context: Context):
    strategy = FedAvg(
        min_fit_clients=5,
        fraction_evaluate=0.0,
        initial_parameters=params,
        on_fit_config_fn=fit_config,
    )
    config = ServerConfig(num_rounds=3)
    return ServerAppComponents(
        strategy=strategy,
        config=config,
    )

### **Create an Instance of ServerApp**


In [ ]:
server = ServerApp(server_fn=server_fn)


---

## **Step 4: Implement Client-Side Training and Evaluation**
The client will receive training configuration and execute local model updates.


### **Define Flower Client**


In [ ]:
class FlowerClient(NumPyClient):
    def __init__(self, net, trainloader, testloader):
        self.net = net
        self.trainloader = trainloader
        self.testloader = testloader

    def fit(self, parameters, config):
        set_weights(self.net, parameters)

        epochs = config["local_epochs"]
        log(INFO, f"Client trains for {epochs} epochs")
        train_model(self.net, self.trainloader, epochs)

        return get_weights(self.net), len(self.trainloader), {}

    def evaluate(self, parameters, config):
        set_weights(self.net, parameters)
        loss, accuracy = evaluate_model(self.net, self.testloader)
        return loss, len(self.testloader), {"accuracy": accuracy}

### **Create the Client Function and Client App**


In [ ]:
def client_fn(context: Context) -> Client:
    net = SimpleModel()
    partition_id = int(context.node_config["partition-id"])
    trainloader, testloader = load_data(partition_id=partition_id)
    return FlowerClient(net, trainloader, testloader).to_client()

client = ClientApp(client_fn)

---

## **Step 5: Run Federated Training Simulation**

In [ ]:
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=5,
    backend_config=backend_setup,
)

DEBUG:flwr:Asyncio event loop already running.
INFO : Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO : 
INFO : [INIT]
INFO : Using initial global parameters provided by strategy
INFO : Evaluating initial global parameters
INFO : 
INFO : [ROUND 1]
INFO : configure_fit: strategy sampled 5 clients (out of 5)
(pid=2714) 2025-02-08 21:44:14.539095: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=2714) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=2714) E0000 00:00:1739051054.565419    2714 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=2714) E0000 00:00:1739051054.575272    2714 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has